# EfficientNet-B4 From Scratch (PyTorch)

This notebook implements **EfficientNet-B4** from scratch in PyTorch, without importing any prebuilt EfficientNet model from `torchvision` or elsewhere.

The notebook includes:
- model components (Swish, DropConnect, SE, MBConv),
- stage scaling for B4,
- full model assembly,
- training + validation loops,
- evaluation/inference/checkpointing utilities.

## 1) Environment Setup and Library Imports (PyTorch)

In [1]:
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Device: cuda


## 2) Global Configuration for EfficientNet-B4

In [2]:
@dataclass
class Config:
    # EfficientNet-B4 scaling
    width_coefficient: float = 1.4
    depth_coefficient: float = 1.8
    input_resolution: int = 224
    dropout_rate: float = 0.4
    drop_connect_rate: float = 0.2
    depth_divisor: int = 8

    # Data
    dataset_name: str = "cifar10"  # one of: mnist, cifar10, cifar100
    data_root: str = "./data"

    # Training
    num_classes: int = 10
    batch_size: int = 64
    epochs: int = 30
    learning_rate: float = 3e-4
    weight_decay: float = 1e-4
    num_workers: int = 2
    amp: bool = True
    label_smoothing: float = 0.1
    grad_clip_norm: float = 1.0

    # Attention (inside MBConv block)
    attention_type: str = "se"  # one of: se, eca, none

    # Knowledge distillation
    use_distillation: bool = False
    distill_alpha: float = 0.5
    distill_temperature: float = 4.0

    # XAI
    xai_target_layer: str = "head"


CFG = Config()
print(CFG)

Config(width_coefficient=1.4, depth_coefficient=1.8, input_resolution=224, dropout_rate=0.4, drop_connect_rate=0.2, depth_divisor=8, dataset_name='cifar10', data_root='./data', num_classes=10, batch_size=64, epochs=30, learning_rate=0.0003, weight_decay=0.0001, num_workers=2, amp=True, label_smoothing=0.1, grad_clip_norm=1.0, attention_type='se', use_distillation=False, distill_alpha=0.5, distill_temperature=4.0, xai_target_layer='head')


## 3) Data Loading and Image Preprocessing Pipeline

In [3]:
DATA_STATS = {
    "mnist": {"mean": [0.1307, 0.1307, 0.1307], "std": [0.3081, 0.3081, 0.3081], "num_classes": 10},
    "cifar10": {"mean": [0.4914, 0.4822, 0.4465], "std": [0.2470, 0.2435, 0.2616], "num_classes": 10},
    "cifar100": {"mean": [0.5071, 0.4867, 0.4408], "std": [0.2675, 0.2565, 0.2761], "num_classes": 100},
}


def _build_transforms(cfg: Config):
    stats = DATA_STATS[cfg.dataset_name]
    mean, std = stats["mean"], stats["std"]

    train_tfms = [
        transforms.Resize((cfg.input_resolution, cfg.input_resolution)),
    ]
    if cfg.dataset_name == "mnist":
        train_tfms.append(transforms.Grayscale(num_output_channels=3))

    train_tfms.extend([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    val_tfms = [
        transforms.Resize((cfg.input_resolution, cfg.input_resolution)),
    ]
    if cfg.dataset_name == "mnist":
        val_tfms.append(transforms.Grayscale(num_output_channels=3))

    val_tfms.extend([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    return transforms.Compose(train_tfms), transforms.Compose(val_tfms)


def build_dataloaders(cfg: Config):
    dataset_name = cfg.dataset_name.lower()
    assert dataset_name in DATA_STATS, f"Unsupported dataset: {dataset_name}"
    cfg.dataset_name = dataset_name
    cfg.num_classes = DATA_STATS[dataset_name]["num_classes"]

    train_transforms, val_transforms = _build_transforms(cfg)

    if dataset_name == "mnist":
        train_ds = datasets.MNIST(root=cfg.data_root, train=True, download=True, transform=train_transforms)
        val_ds = datasets.MNIST(root=cfg.data_root, train=False, download=True, transform=val_transforms)
    elif dataset_name == "cifar10":
        train_ds = datasets.CIFAR10(root=cfg.data_root, train=True, download=True, transform=train_transforms)
        val_ds = datasets.CIFAR10(root=cfg.data_root, train=False, download=True, transform=val_transforms)
    else:
        train_ds = datasets.CIFAR100(root=cfg.data_root, train=True, download=True, transform=train_transforms)
        val_ds = datasets.CIFAR100(root=cfg.data_root, train=False, download=True, transform=val_transforms)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
    )
    return train_loader, val_loader


train_loader, val_loader = build_dataloaders(CFG)
print(f"Dataset: {CFG.dataset_name}, classes: {CFG.num_classes}")
print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

100%|██████████| 170M/170M [00:02<00:00, 78.3MB/s] 


Dataset: cifar10, classes: 10
Train samples: 50000
Val samples: 10000
Train batches: 782, Val batches: 157


## 4) Core Utility Layers: Swish, Drop Connect, and Same Padding

In [4]:
class Swish(nn.Module):
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.sigmoid(x)


def drop_connect(x: torch.Tensor, drop_prob: float, training: bool) -> torch.Tensor:
    if not training or drop_prob == 0.0:
        return x
    keep_prob = 1.0 - drop_prob
    batch_size = x.shape[0]
    random_tensor = keep_prob + torch.rand((batch_size, 1, 1, 1), dtype=x.dtype, device=x.device)
    binary_mask = random_tensor.floor()
    return (x / keep_prob) * binary_mask


class Conv2dSame(nn.Conv2d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, groups=1, bias=False):
        super().__init__(
            in_channels,
            out_channels,
            kernel_size,
            stride=stride,
            padding=0,
            groups=groups,
            bias=bias,
        )

    def _get_same_padding(self, x: torch.Tensor):
        ih, iw = x.shape[-2:]
        kh, kw = self.kernel_size
        sh, sw = self.stride
        dh, dw = self.dilation

        oh = math.ceil(ih / sh)
        ow = math.ceil(iw / sw)

        pad_h = max((oh - 1) * sh + (kh - 1) * dh + 1 - ih, 0)
        pad_w = max((ow - 1) * sw + (kw - 1) * dw + 1 - iw, 0)

        return [
            pad_w // 2,
            pad_w - pad_w // 2,
            pad_h // 2,
            pad_h - pad_h // 2,
        ]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pad = self._get_same_padding(x)
        if any(pad):
            x = F.pad(x, pad)
        return F.conv2d(x, self.weight, self.bias, self.stride, 0, self.dilation, self.groups)

## 5) Squeeze-and-Excitation Block Implementation

In [5]:
class SqueezeExcitation(nn.Module):
    def __init__(self, in_channels: int, squeeze_channels: int):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.reduce = nn.Conv2d(in_channels, squeeze_channels, kernel_size=1)
        self.expand = nn.Conv2d(squeeze_channels, in_channels, kernel_size=1)
        self.act = Swish()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        scale = self.pool(x)
        scale = self.reduce(scale)
        scale = self.act(scale)
        scale = self.expand(scale)
        scale = torch.sigmoid(scale)
        return x * scale

## 6) MBConv Block Implementation from Scratch

In [6]:
class ECABlock(nn.Module):
    def __init__(self, channels: int, k_size: int = 3):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Conv1d(1, 1, kernel_size=k_size, padding=(k_size - 1) // 2, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.pool(x)
        y = y.squeeze(-1).transpose(-1, -2)
        y = self.conv(y)
        y = y.transpose(-1, -2).unsqueeze(-1)
        y = torch.sigmoid(y)
        return x * y


class MBConvBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        stride: int,
        expand_ratio: int,
        se_ratio: float = 0.25,
        drop_connect_rate: float = 0.0,
        attention_type: str = "se",
    ):
        super().__init__()
        self.use_residual = stride == 1 and in_channels == out_channels
        self.drop_connect_rate = drop_connect_rate
        expanded_channels = in_channels * expand_ratio
        squeeze_channels = max(1, int(in_channels * se_ratio))

        layers = []

        # Expansion
        if expand_ratio != 1:
            layers.extend([
                Conv2dSame(in_channels, expanded_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(expanded_channels),
                Swish(),
            ])

        # Depthwise
        layers.extend([
            Conv2dSame(
                expanded_channels,
                expanded_channels,
                kernel_size=kernel_size,
                stride=stride,
                groups=expanded_channels,
                bias=False,
            ),
            nn.BatchNorm2d(expanded_channels),
            Swish(),
        ])

        # Optional attention
        if attention_type == "se":
            layers.append(SqueezeExcitation(expanded_channels, squeeze_channels=squeeze_channels))
        elif attention_type == "eca":
            layers.append(ECABlock(expanded_channels))

        # Projection
        layers.extend([
            Conv2dSame(expanded_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
        ])

        self.block = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.block(x)
        if self.use_residual:
            out = drop_connect(out, self.drop_connect_rate, self.training)
            out = out + x
        return out

## 7) Stage Builder with EfficientNet-B4 Repeats and Channels

In [7]:
def round_filters(filters: int, width_coefficient: float, depth_divisor: int = 8) -> int:
    filters *= width_coefficient
    min_depth = depth_divisor
    new_filters = max(
        min_depth,
        int(filters + depth_divisor / 2) // depth_divisor * depth_divisor,
    )
    if new_filters < 0.9 * filters:
        new_filters += depth_divisor
    return int(new_filters)


def round_repeats(repeats: int, depth_coefficient: float) -> int:
    return int(math.ceil(depth_coefficient * repeats))


# Base EfficientNet block args: (expand_ratio, channels, repeats, stride, kernel_size)
BASE_BLOCKS = [
    (1, 16, 1, 1, 3),
    (6, 24, 2, 2, 3),
    (6, 40, 2, 2, 5),
    (6, 80, 3, 2, 3),
    (6, 112, 3, 1, 5),
    (6, 192, 4, 2, 5),
    (6, 320, 1, 1, 3),
]


def build_block_specs(cfg: Config):
    specs = []
    for expand_ratio, channels, repeats, stride, kernel in BASE_BLOCKS:
        out_channels = round_filters(channels, cfg.width_coefficient, cfg.depth_divisor)
        num_repeats = round_repeats(repeats, cfg.depth_coefficient)
        specs.append((expand_ratio, out_channels, num_repeats, stride, kernel))
    return specs


block_specs = build_block_specs(CFG)
print("B4 block specs:")
for s in block_specs:
    print(s)

B4 block specs:
(1, 24, 2, 1, 3)
(6, 32, 4, 2, 3)
(6, 56, 4, 2, 5)
(6, 112, 6, 2, 3)
(6, 160, 6, 1, 5)
(6, 272, 8, 2, 5)
(6, 448, 2, 1, 3)


## 8) EfficientNet-B4 Model Class (No Prebuilt Model Imports)

In [8]:
class EfficientNet(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg

        stem_out = round_filters(32, cfg.width_coefficient, cfg.depth_divisor)
        self.stem = nn.Sequential(
            Conv2dSame(3, stem_out, kernel_size=3, stride=2, bias=False),
            nn.BatchNorm2d(stem_out),
            Swish(),
        )

        blocks = []
        in_channels = stem_out
        block_specs = build_block_specs(cfg)
        total_blocks = sum(spec[2] for spec in block_specs)
        block_index = 0

        for expand_ratio, out_channels, repeats, stride, kernel_size in block_specs:
            for i in range(repeats):
                block_stride = stride if i == 0 else 1
                block_drop_rate = cfg.drop_connect_rate * block_index / max(1, total_blocks - 1)
                blocks.append(
                    MBConvBlock(
                        in_channels=in_channels,
                        out_channels=out_channels,
                        kernel_size=kernel_size,
                        stride=block_stride,
                        expand_ratio=expand_ratio,
                        drop_connect_rate=block_drop_rate,
                        attention_type=cfg.attention_type,
                    )
                )
                in_channels = out_channels
                block_index += 1

        self.blocks = nn.Sequential(*blocks)

        head_in = in_channels
        head_out = round_filters(1280, cfg.width_coefficient, cfg.depth_divisor)
        self.head = nn.Sequential(
            Conv2dSame(head_in, head_out, kernel_size=1, bias=False),
            nn.BatchNorm2d(head_out),
            Swish(),
        )

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(cfg.dropout_rate)
        self.classifier = nn.Linear(head_out, cfg.num_classes)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x


def EfficientNetB4(cfg: Optional[Config] = None, num_classes: Optional[int] = None) -> EfficientNet:
    if cfg is None:
        cfg = Config()
    if num_classes is not None:
        cfg.num_classes = num_classes
    return EfficientNet(cfg)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


model = EfficientNetB4(cfg=CFG).to(DEVICE)
print(model.__class__.__name__)
print(f"Trainable params: {count_parameters(model):,}")
print(f"Attention mode: {CFG.attention_type}")

EfficientNet
Trainable params: 17,566,546
Attention mode: se


In [9]:
# Layer audit: verify depthwise convs and stage output shapes
conv_issues = []
for name, m in model.named_modules():
    if isinstance(m, nn.Conv2d) and m.kernel_size != (1, 1):
        is_depthwise = (m.groups == m.in_channels == m.out_channels)
        if "blocks" in name and not is_depthwise and m.groups != 1:
            conv_issues.append((name, m.in_channels, m.out_channels, m.groups, m.kernel_size))

print(f"Depthwise/grouped conv issues found: {len(conv_issues)}")
for issue in conv_issues[:10]:
    print(issue)

# Stage shape trace
shape_trace = {}
hooks = []

def _hook(name):
    def fn(_, __, out):
        shape_trace[name] = tuple(out.shape)
    return fn

hooks.append(model.stem.register_forward_hook(_hook("stem")))
hooks.append(model.blocks.register_forward_hook(_hook("blocks")))
hooks.append(model.head.register_forward_hook(_hook("head")))

model.eval()
with torch.no_grad():
    x = torch.randn(1, 3, CFG.input_resolution, CFG.input_resolution).to(DEVICE)
    y = model(x)

for h in hooks:
    h.remove()

print("Shape trace:", shape_trace)
print("Classifier output:", tuple(y.shape))

Depthwise/grouped conv issues found: 0
Shape trace: {'stem': (1, 48, 112, 112), 'blocks': (1, 448, 7, 7), 'head': (1, 1792, 7, 7)}
Classifier output: (1, 10)


## 9) Training Setup: Loss, Optimizer, Scheduler, and AMP

In [10]:
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.epochs)
scaler = torch.cuda.amp.GradScaler(enabled=(CFG.amp and DEVICE.type == "cuda"))


def distillation_loss(
    student_logits: torch.Tensor,
    targets: torch.Tensor,
    base_criterion: nn.Module,
    teacher_logits: Optional[torch.Tensor] = None,
    alpha: float = 0.5,
    temperature: float = 4.0,
) -> torch.Tensor:
    ce = base_criterion(student_logits, targets)
    if teacher_logits is None:
        return ce

    kd = F.kl_div(
        F.log_softmax(student_logits / temperature, dim=1),
        F.softmax(teacher_logits / temperature, dim=1),
        reduction="batchmean",
    ) * (temperature ** 2)

    return (1.0 - alpha) * ce + alpha * kd


def topk_accuracy(logits: torch.Tensor, targets: torch.Tensor, topk=(1, 5)) -> List[torch.Tensor]:
    with torch.no_grad():
        maxk = max(topk)
        _, pred = logits.topk(maxk, dim=1, largest=True, sorted=True)
        pred = pred.t()
        correct = pred.eq(targets.view(1, -1).expand_as(pred))
        results = []
        for k in topk:
            correct_k = correct[:k].reshape(-1).float().sum(0)
            results.append(correct_k * (100.0 / targets.size(0)))
    return results

/tmp/ipykernel_55/1238209896.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(CFG.amp and DEVICE.type == "cuda"))


## 10) Training and Validation Loops

In [11]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device,
    cfg: Config,
    epoch: int,
    total_epochs: int,
    teacher_model: Optional[nn.Module] = None,
):
    model.train()
    if teacher_model is not None:
        teacher_model.eval()

    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    total_samples = 0

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} [Train]", leave=False)
    for images, targets in pbar:
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(cfg.amp and device.type == "cuda")):
            student_logits = model(images)
            teacher_logits = None
            if cfg.use_distillation and teacher_model is not None:
                with torch.no_grad():
                    teacher_logits = teacher_model(images)

            loss = distillation_loss(
                student_logits=student_logits,
                targets=targets,
                base_criterion=criterion,
                teacher_logits=teacher_logits,
                alpha=cfg.distill_alpha,
                temperature=cfg.distill_temperature,
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
        scaler.step(optimizer)
        scaler.update()

        top1, top5 = topk_accuracy(student_logits, targets, topk=(1, 5))
        bs = images.size(0)
        total_loss += loss.item() * bs
        total_top1 += top1.item() * bs / 100.0
        total_top5 += top5.item() * bs / 100.0
        total_samples += bs

        pbar.set_postfix({
            "batch_loss": f"{loss.item():.4f}",
            "avg_loss": f"{(total_loss / total_samples):.4f}",
        })

    return {
        "loss": total_loss / total_samples,
        "top1": 100.0 * total_top1 / total_samples,
        "top5": 100.0 * total_top5 / total_samples,
    }


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, epoch=None, total_epochs=None):
    model.eval()
    total_loss = 0.0
    total_top1 = 0.0
    total_top5 = 0.0
    total_samples = 0

    if epoch is None or total_epochs is None:
        desc = "Validation"
    else:
        desc = f"Epoch {epoch}/{total_epochs} [Val]"
    pbar = tqdm(loader, desc=desc, leave=False)

    for images, targets in pbar:
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, targets)
        top1, top5 = topk_accuracy(logits, targets, topk=(1, 5))

        bs = images.size(0)
        total_loss += loss.item() * bs
        total_top1 += top1.item() * bs / 100.0
        total_top5 += top5.item() * bs / 100.0
        total_samples += bs

        pbar.set_postfix({
            "batch_loss": f"{loss.item():.4f}",
            "avg_loss": f"{(total_loss / total_samples):.4f}",
        })

    return {
        "loss": total_loss / total_samples,
        "top1": 100.0 * total_top1 / total_samples,
        "top5": 100.0 * total_top5 / total_samples,
    }


def fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    scheduler,
    scaler,
    device,
    cfg: Config,
    teacher_model: Optional[nn.Module] = None,
):
    best_top1 = -1.0
    history = []

    for epoch in range(1, cfg.epochs + 1):
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            scaler=scaler,
            device=device,
            cfg=cfg,
            epoch=epoch,
            total_epochs=cfg.epochs,
            teacher_model=teacher_model,
        )
        val_metrics = validate_one_epoch(
            model, val_loader, criterion, device, epoch, cfg.epochs
        )
        scheduler.step()

        history.append((train_metrics, val_metrics))

        tqdm.write(
            f"Epoch [{epoch}/{cfg.epochs}] | "
            f"Train Loss: {train_metrics['loss']:.4f}, Top1: {train_metrics['top1']:.2f}, Top5: {train_metrics['top5']:.2f} | "
            f"Val Loss: {val_metrics['loss']:.4f}, Top1: {val_metrics['top1']:.2f}, Top5: {val_metrics['top5']:.2f}"
        )

        if val_metrics["top1"] > best_top1:
            best_top1 = val_metrics["top1"]
            torch.save(model.state_dict(), "efficientnet_b4_best.pth")

    torch.save(model.state_dict(), "efficientnet_b4_last.pth")
    return history

In [12]:
# Optional teacher model for knowledge distillation. Keep None for baseline training.
teacher_model = None

# Example (when you have a trained teacher checkpoint):
# teacher_model = EfficientNetB4(cfg=CFG).to(DEVICE)
# teacher_model.load_state_dict(torch.load("teacher_best.pth", map_location=DEVICE))
# teacher_model.eval()

history = fit(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    scaler=scaler,
    device=DEVICE,
    cfg=CFG,
    teacher_model=teacher_model,
)

Epoch 1/30 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

/tmp/ipykernel_55/1169509695.py:27: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(cfg.amp and device.type == "cuda")):


Epoch 1/30 [Val]:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch [1/30] | Train Loss: 1.7890, Top1: 38.80, Top5: 86.99 | Val Loss: 1.4471, Top1: 57.37, Top5: 95.04


Epoch 2/30 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

...

Epoch 29/30 [Val]:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch [29/30] | Train Loss: 0.5119, Top1: 99.69, Top5: 99.99 | Val Loss: 0.7229, Top1: 92.45, Top5: 98.97


Epoch 30/30 [Train]:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 30/30 [Val]:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch [30/30] | Train Loss: 0.5120, Top1: 99.68, Top5: 99.99 | Val Loss: 0.7218, Top1: 92.51, Top5: 98.95


## 11) XAI Utility: Grad-CAM Heatmap

In [13]:
class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self.hook_handles = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(_, __, output):
            self.activations = output.detach()

        def backward_hook(_, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.hook_handles.append(self.target_layer.register_forward_hook(forward_hook))
        self.hook_handles.append(self.target_layer.register_full_backward_hook(backward_hook))

    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()

    def __call__(self, input_tensor: torch.Tensor, class_idx: Optional[int] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        self.model.eval()
        logits = self.model(input_tensor)

        if class_idx is None:
            class_idx = int(torch.argmax(logits, dim=1).item())

        self.model.zero_grad(set_to_none=True)
        score = logits[:, class_idx].sum()
        score.backward(retain_graph=True)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(
            cam, size=input_tensor.shape[-2:], mode="bilinear", align_corners=False
        )
        cam = cam.squeeze(0).squeeze(0)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        return cam.detach().cpu(), logits.detach().cpu()


def get_xai_target_layer(model: EfficientNet, layer_name: str) -> nn.Module:
    if layer_name == "stem":
        return model.stem
    if layer_name == "blocks":
        return model.blocks
    return model.head


@torch.no_grad()
def denormalize_for_plot(x: torch.Tensor, dataset_name: str) -> torch.Tensor:
    stats = DATA_STATS[dataset_name]
    mean = torch.tensor(stats["mean"], device=x.device).view(1, 3, 1, 1)
    std = torch.tensor(stats["std"], device=x.device).view(1, 3, 1, 1)
    return x * std + mean


def show_gradcam_for_batch(
    model: EfficientNet,
    images: torch.Tensor,
    labels: torch.Tensor,
    cfg: Config,
    max_images: int = 4,
):
    images = images[:max_images].to(DEVICE)
    labels = labels[:max_images]

    target_layer = get_xai_target_layer(model, cfg.xai_target_layer)
    gradcam = GradCAM(model, target_layer)

    model.eval()
    fig, axes = plt.subplots(max_images, 3, figsize=(10, 3 * max_images))
    if max_images == 1:
        axes = [axes]

    for i in range(max_images):
        cam, logits = gradcam(images[i : i + 1])
        pred = int(torch.argmax(logits, dim=1).item())
        denorm_img = denormalize_for_plot(images[i : i + 1], cfg.dataset_name)[0].detach().cpu()
        denorm_img = denorm_img.permute(1, 2, 0).clamp(0, 1).numpy()

        axes[i][0].imshow(denorm_img)
        axes[i][0].set_title(f"Image | y={int(labels[i])}")
        axes[i][0].axis("off")

        axes[i][1].imshow(cam.numpy(), cmap="jet")
        axes[i][1].set_title("Grad-CAM")
        axes[i][1].axis("off")

        axes[i][2].imshow(denorm_img)
        axes[i][2].imshow(cam.numpy(), cmap="jet", alpha=0.45)
        axes[i][2].set_title(f"Overlay | pred={pred}")
        axes[i][2].axis("off")

    plt.tight_layout()
    gradcam.remove_hooks()


# Example XAI call after training:
# xai_images, xai_labels = next(iter(val_loader))
# show_gradcam_for_batch(model, xai_images, xai_labels, CFG, max_images=4)

## 12) Evaluation, Inference, and Checkpoint Export

In [14]:
@torch.no_grad()
def run_inference(model: nn.Module, images: torch.Tensor, device: torch.device):
    model.eval()
    images = images.to(device)
    logits = model(images)
    probs = torch.softmax(logits, dim=1)
    conf, pred = probs.max(dim=1)
    return pred.cpu(), conf.cpu()


# Load saved checkpoint example (after training):
# model.load_state_dict(torch.load("efficientnet_b4_best.pth", map_location=DEVICE))

# Final validation metrics example:
val_metrics = validate_one_epoch(model, val_loader, criterion, DEVICE)
print("Validation metrics (current weights):", val_metrics)

# Batch inference example from val loader:
images, labels = next(iter(val_loader))
pred, conf = run_inference(model, images[:4], DEVICE)
print("Predicted classes:", pred.tolist())
print("Confidences:", [round(x, 4) for x in conf.tolist()])

Validation:   0%|          | 0/157 [00:00<?, ?it/s]

Validation metrics (current weights): {'loss': 0.7218165683746338, 'top1': 92.51, 'top5': 98.95}
Predicted classes: [3, 8, 8, 0]
Confidences: [0.8985, 0.9014, 0.8626, 0.8342]


## 13) Sanity Check: Forward Pass with [2, 3, H, W]

In [15]:
# Kaggle checkpoint cell: save + reload model for future use
import os
import torch
from dataclasses import asdict

SAVE_DIR = "/kaggle/working/mydata"
CKPT_PATH = "/kaggle/working/mydata/efficientnet_b4_mnist_checkpoint.pth"
os.makedirs(SAVE_DIR, exist_ok=True)

checkpoint = {
    "model_state_dict": model.state_dict(),
    "cfg": asdict(CFG),
    "num_classes": CFG.num_classes,
}

if "optimizer" in globals():
    checkpoint["optimizer_state_dict"] = optimizer.state_dict()
if "scheduler" in globals():
    checkpoint["scheduler_state_dict"] = scheduler.state_dict()

torch.save(checkpoint, CKPT_PATH)
print(f"Saved checkpoint: {CKPT_PATH}")

# Example reload (same notebook or later notebook with model classes defined):
loaded = torch.load(CKPT_PATH, map_location=DEVICE)
loaded_cfg = Config(**loaded["cfg"])
loaded_model = EfficientNet(loaded_cfg).to(DEVICE)
loaded_model.load_state_dict(loaded["model_state_dict"])
loaded_model.eval()
print("Reload successful. Model is ready for inference.")

Saved checkpoint: /kaggle/working/mydata/efficientnet_b4_mnist_checkpoint.pth
Reload successful. Model is ready for inference.


In [16]:
model.eval()
dummy = torch.randn(2, 3, CFG.input_resolution, CFG.input_resolution).to(DEVICE)
with torch.no_grad():
    out = model(dummy)

print("Dummy input shape:", tuple(dummy.shape))
print("Output shape:", tuple(out.shape))  # expected: (2, num_classes)

Dummy input shape: (2, 3, 224, 224)
Output shape: (2, 10)
